### [practice on LLM chapter 1-2, 2-1]

#### chapter 1-2

### 문제 1-1 : 역사적 사실 검증관 프롬프트 작성하기]

<aside>

**문제 설명**

---

최근 인공지능이 존재하지 않는 가짜 역사를 진짜처럼 설명하여 혼란을 주는 사례가 늘고 있습니다. 우리는 학습 데이터를 벗어난 질문에 대해 솔직하게 모른다고 답변하는 안전한 정보 검증 인공지능을 구현해야 합니다. 

</aside>

<aside>

**요구 사항**

---

1. ChatPromptTemplate을 사용하여 프롬프트를 구성하세요. 시스템 역할에는 객관적 사실만 전달하고 모르는 내용은 확인할 수 없다고 명확히 답변하도록 지시하세요.
2. ChatOpenAI 모델을 선언할 때 단어 선택의 무작위성을 없애고 일관성을 극대화하기 위해 temperature 값을 0으로 고정하세요.
3. 순수 텍스트를 추출하는 StrOutputParser를 추가하고 파이프 연산자를 통해 실행 체인을 하나로 결합하세요.
4. 사용자 질문으로 1999년 조선시대에 발명된 스마트폰의 기능에 대해 설명해줘 라는 가짜 역사 질문을 입력하여 코드를 실행하세요.
</aside>

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

prompt_template = ChatPromptTemplate([
    ("system", "당신은 학습 데이터를 벗어난 질문에 대해 솔직하게 모른다고 답변하는 안전한 정보 검증 인공지능입니다. 객관적 사실만 전달하고, 모르는 내용은 '확인할 수 없습니다'라고 명확히 답변하세요."),
    ("user", "{user_question}")
])

model = init_chat_model(
    model_provider="ollama",
    model="mistral",
    temperature=0
)

parser = StrOutputParser()

chain = prompt_template | model | parser

question = "1999년 조선시대에 발명된 스마트폰의 기능에 대해 설명해줘."

response = chain.invoke({"user_question": question})

response

' 조선시대는 1392년부터 1910년까지 조선 왕조가 지배한 시기이며, 이 시기에는 스마트폰이 발명되지 않았습니다. 스마트폰은 21세기 초반에 발명되었으며, 이 기기는 전화, 인터넷 브라우저, 메일, 앱 등 다양한 기능을 가지고 있습니다. 조선시대에 발명된 기기는 전화기와 같은 기능을 가지고 있었지만, 스마트폰과는 다릅니다. 조선시대에 발명된 기기에 대한 자세한 정보는 조선 왕조 기록을 참고하시기 바랍니다.'

## 필수 2 : 민감 정보 유출 방지를 위한 데이터 마스킹 실습

<aside>

질문 내용이 외부 서버로 전송되기 전에 고객의 신용카드 번호와 같은 민감한 데이터가 유출되지 않도록 가려주는 전처리 기법을 실습합니다. 

</aside>

### 문제 2-1 :  신용카드 번호 마스킹 함수 구현하기

<aside>

**문제 설명**

---

결제 시스템 오류를 묻는 고객의 메시지에는 16자리의 신용카드 번호가 그대로 노출되는 경우가 많습니다. 원본 데이터를 그대로 인공지능에게 전달하면 외부 유출의 위험이 있으므로 전송 전에 숫자들을 안전한 문구로 치환해야 합니다.

</aside>

<aside>

**요구사항**

---

- 파이썬의 re 모듈을 불러오고 텍스트를 입력받아 문자열을 반환하는 함수를 정의하세요.
- 0000-0000-0000-0000 형태의 16자리 카드 번호를 감지할 수 있도록 정규표현식 패턴을 작성하세요.
- 입력된 문장에서 해당 패턴이 발견되면 카드번호 보호 라는 안전한 대체 문구로 치환하여 반환하도록 코드를 완성하세요.
- 결제 오류가 발생했습니다 제 카드 번호는 1234-5678-9012-3456 입니다 확인 부탁드립니다 라는 문장을 함수에 넣고 실행하여 결과를 출력하세요.
</aside>

<aside>

출력 및 검증 방법

---

1. 터미널에서 코드를 실행하여 원본 문장의 16자리 숫자가 보호 문구로 정확하게 치환되어 출력되는지 확인합니다.
2. 안전한 텍스트로 변환된 결과가 나타난 터미널 화면을 캡처합니다.
3. 해당 출력 화면 캡처와 작성된 소스코드를 제출하세요.
</aside>

In [8]:
import  # 정규표현식 사용을 위한 모듈 re import

def mask_personal_information(text: str) -> str: 
# type hint. text:str (text 매개변수는 str이 들어올 것)
# -> str(이 함수는 str 반환할 예정)
    credit_pattern = re.compile(r'\d{4}-\d{4}-\d{4}-\d{4}')
    # 카드번호 (4자리 숫자 연속 4개) 형식 정규표현식 만들기
    # credit_pattern에 저장

    masked = credit_pattern.sub("[카드번호 보호]", text)
    # text안에서 credit_pattern과 일치하는 부분 찾기
    # 일치하는 부분 찾으면 [카드번호 보호]로 바꾸기
    # .sub(바꿀문자열, 대상문자열)
    return masked

user_input = "신용카드 번호는 1234-5678-9011-1234 입니다."
mask_personal_information(user_input)

'신용카드 번호는 [카드번호 보호] 입니다.'

## 심화 1 : 안전 점검 대시보드 웹 애플리케이션 제작

<aside>

지금까지 배운 모델 제어 기술과 데이터 마스킹 기술을 하나로 묶어 스트림릿 기반의 안전한 인공지능 웹 화면을 완성해 봅니다.

</aside>

### 문제 3-1 : 스트림릿 기반 데이터 보호 질의 시스템 만들기

<aside>

**문제 설명**

---

동료들이 인공지능을 사용할 때 실수로 사내 기밀을 유출하는 것을 막아야 합니다. 사용자가 보안 서약 항목을 직접 체크해야만 인공지능에게 질문을 보낼 수 있도록 제어하는 안전 점검 화면을 구축해야 합니다.

</aside>

<aside>

**요구사항**

---

- 스트림릿 라이브러리를 활용하여 웹 페이지의 제목과 목적을 안내하는 문구를 화면 상단에 출력하세요.
- 사용자가 필수적으로 동의해야 하는 안전 점검 체크박스 1개를 만드세요. 항목 내용은 고객의 개인정보나 금융 정보를 포함하지 않았습니다 로 지정하세요.
- 사용자로부터 질문 내용을 입력받을 수 있는 텍스트 영역을 화면에 추가하세요.
- 실행 버튼을 만들고 체크박스가 선택되지 않은 상태에서 버튼을 누르면 에러 알림을 화면에 띄우도록 조건문을 구성하세요..
- 정상적으로 체크가 완료된 상태에서 버튼을 누르면 필수 2에서 만든 마스킹 함수를 실행한 후 인공지능 모델을 호출하여 안전하게 답변을 화면에 출력하도록 하세요.
</aside>

<aside>

출력 및 검증 방법

---

1. 터미널에서 uv run streamlit run 명령어를 입력하여 웹 서버를 실행합니다. 
2. 브라우저에 나타난 웹 화면에서 체크박스를 누르지 않고 실행 버튼을 눌렀을 때 경고 알림이 뜨는 화면을 캡처합니다. 
3. 체크박스를 누른 후 카드 번호가 포함된 질문을 입력하여 마스킹과 질의 응답이 정상적으로 완료된 화면을 캡처합니다. 
4. 해당 출력 화면 캡처와 작성된 소스코드를 제출하세요.
</aside>

#### chapter 2-1

## 필수 1 : 나의 전담 여행 가이드 만들기

<aside>

인공지능에게 여행 가이드 역할을 부여하고, 가변적인 사용자 입력을 통해 원하는 도시의 정보를 얻는 실습입니다.

</aside>

### 문제 1-1 : 특정 도시에 맞춤화된 추천 답변 생성하기

<aside>

**문제 설명**

---

인공지능이 단순한 백과사전처럼 답하지 않고 실제 가이드처럼 친절하게 답변하도록 전역 정책을 설정해야 합니다. 사용자가 여행할 도시를 매번 다르게 입력할 수 있도록 구조를 설계합니다. 

</aside>

<aside>

**요구 사항**

---

1. 프롬프트 템플릿을 생성합니다.
2. 시스템 메시지에 당신은 친절한 여행 가이드입니다. 방문하는 도시의 대표 관광지 1곳을 추천해주세요.라고 모델의 페르소나를 설정합니다.
3. 사용자 메시지에 이번 주말에 {city}로 여행을 갑니다.라고 작성하여 가변적 입력 변수를 정의합니다.
4. 체인을 연결한 후, city 변수에 부산을 전달하여 모델을 실행합니다. 
</aside>

<aside>

출력 및 검증 방법

---

1. VSCode에서 새로운 파이썬 파일을 생성합니다.
2. 요구 사항에 맞추어 템플릿과 체인을 구성하는 코드를 작성하고 실행합니다.
3. 터미널 창에 부산 관광지가 추천된 결과 텍스트가 나타난 화면을 캡처합니다.
4. 해당 출력 화면 캡처와 작성된 소스코드를 제출하세요.
</aside>

In [12]:
from langchain.chat_models import init_chat_model # 사용할 채팅 모델 초기화 함수
from langchain_core.output_parsers import StrOutputParser # 모델 응답을 str으로 바꿔줌
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage # langchain에서 사용하는 메시지 객체 import
from langchain_core.prompts import ChatPromptTemplate   

model = init_chat_model(
    model_provider="ollama",
    model="coolsoon/kanana-1.5-8b"
)

parser = StrOutputParser()


prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 여행 가이드입니다. 방문하는 도시의 대표 관광지 1곳을 추천해주세요."),
    ("user", "이번 주말에 {city}로 여행을 갑니다.")
])

chain = prompt_template | model | parser

response = chain.invoke({"city": "부산"})
response

'부산은 활기찬 해안 도시로 다양한 명소가 있습니다.  \n주말에 부산을 방문하신다면 **부산 해운대 해수욕장**을 추천합니다!\n\n해운대 해수욕장은 부산을 대표하는 해변으로, 넓은 백사장과 맑은 바다가 아름다워 많은 사람들이 찾는 명소입니다.  \n여름엔 시원한 바다에서 수영이나 물놀이를 즐길 수 있고,  \n겨울에도 해변 산책과 일몰 감상이 가능합니다.  \n근처에는 카페, 맛집, 쇼핑거리도 많아 여행하기에 좋습니다.\n\n혹시 해운대 외에 다른 추천지를 원하시면, 관심 있는 장소(예: 바다, 산, 문화, 먹거리 등)를 알려주시면 맞춤 추천도 가능합니다!'

## 필수 2 : 단호한 맞춤법 검사기 만들기

<aside>

인공지능에게 엄격한 출력 제약 조건을 부여하여, 부가 설명 없이 오직 교정된 문장만 출력하도록 통제하는 실습입니다. 

</aside>

### 문제 2-1 :  규칙을 준수하는 맞춤법 교정 시스템 구축하기

<aside>

**문제 설명**

---

비즈니스 문서를 작성할 때 인공지능이 불필요한 인삿말을 덧붙이지 않도록 제어해야 합니다. 시스템 메시지가 사용자 지시보다 우선하여 적용되는 특징을 활용합니다.

</aside>

<aside>

**요구사항**

---

- 프롬프트 템플릿을 생성합니다.
- 시스템 메시지에 당신은 엄격한 국어 교사입니다. 입력된 문장의 맞춤법을 수정하여 결과만 간결하게 출력하세요.라고 출력 제약 조건을 설정합니다.
- 사용자 메시지에 {text} 변수를 넣어 매번 다른 문장을 처리할 수 있도록 작성합니다.
- 체인을 연결하고 text 변수에 오늘 날씨가 참 조으네요.를 전달하여 모델을 실행합니다.
</aside>

<aside>

출력 및 검증 방법

---

1. VSCode에서 새로운 파이썬 파일을 생성합니다.
2. 요구 사항에 맞추어 코드를 작성하고 실행합니다.
3. 터미널 창에 다른 설명 없이 맞춤법이 수정된 문장만 출력된 화면을 캡처합니다.
4. 해당 출력 화면 캡처와 작성된 소스코드를 제출하세요.
</aside>

In [ ]:
from langchain.chat_models import init_chat_model # 사용할 채팅 모델 초기화 함수
from langchain_core.output_parsers import StrOutputParser # 모델 응답을 str으로 바꿔줌
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage # langchain에서 사용하는 메시지 객체 import
from langchain_core.prompts import ChatPromptTemplate   

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 엄격한 국어 교사입니다. 입력된 문장의 맞춤법을 수정하여 결과만 간결하게 출력하세요."),
    ("user", "다음 문장의 문법을 수정하세요:{text}")
])

model = init_chat_model(
    model_provider="ollama",
    model="coolsoon/kanana-1.5-8b"
)

parser = StrOutputParser()

chain = prompt_template | model | parser

response = chain.invoke({"text": "이렇게 하면 않되."})
response


'이렇게 하면 안 돼.'

## 심화 1 : 문맥을 기억하는 단골 식당 점원 만들기

<aside>

이전 대화 턴의 사용자 질문과 어시스턴트 응답을 함께 전달하여, 인공지능이 과거의 맥락을 유지하도록 만드는 실습입니다.

</aside>

### 문제 3-1 : 대화 이력을 활용하여 이어말하기

<aside>

**문제 설명**

---

인공지능은 자체적으로 과거 상태를 저장하지 않으므로, 대명사만 말해도 무엇인지 알 수 있도록 이전 대화 기록을 메시지 목록으로 함께 전달해야 합니다. 

</aside>

<aside>

**요구사항**

---

- 프롬프트 템플릿을 구성할 때 시스템, 사용자, 어시스턴트 메시지를 모두 순서대로 나열합니다.
- 시스템 메시지에 당신은 식당 점원입니다. 친절하게 답변하세요.라고 작성합니다.
- 첫 번째 사용자 메시지에 여기서 가장 인기 있는 메뉴가 무엇인가요?라고 과거 질문을 작성합니다.
- 어시스턴트 메시지에 저희 식당의 최고 인기 메뉴는 치즈 돈까스입니다.라고 직전 대화 턴의 응답 기록을 작성합니다.
- 두 번째 사용자 메시지에 그럼 그걸로 하나 주문할게요. 얼마나 걸리나요?라고 맥락을 참조하는 후속 질문을 작성합니다.
- 체인을 연결하고 추가 입력 변수 없이 모델을 실행하여 결과를 확인합니다.
</aside>

<aside>

출력 및 검증 방법

---

1. VSCode에서 새로운 파이썬 파일을 생성합니다.
2. 세 가지 메시지 유형이 모두 포함되도록 코드를 작성하고 실행합니다.
3. 터미널 창에 치즈 돈까스 주문 접수에 대한 답변이 자연스럽게 이어진 화면을 캡처합니다. 
4. 해당 출력 화면 캡처와 작성된 소스코드를 제출하세요.
</aside>

In [16]:
from langchain.chat_models import init_chat_model # 사용할 채팅 모델 초기화 함수
from langchain_core.output_parsers import StrOutputParser # 모델 응답을 str으로 바꿔줌
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage # langchain에서 사용하는 메시지 객체 import
from langchain_core.prompts import ChatPromptTemplate   

model = init_chat_model(
    model_provider="ollama",
    model="coolsoon/kanana-1.5-8b"
)

parser = StrOutputParser()

chain = model | parser

messages: list[BaseMessage] = [
    SystemMessage("당신은 식당 점원입니다. 친절하게 답변하세요."),
    HumanMessage("여기서 가장 인기있는 메뉴가 무엇인가요?"),
    AIMessage("저희 식당의 최고 인기 멘는 치즈 돈까스입니다."),
    HumanMessage("그럼 그걸로 하나 주문할게요. 얼마나 걸리나요?")
]

response = chain.invoke(messages)
response

'치즈 돈까스는 보통 주문 후 10~15분 정도 걸립니다. 혹시 음료나 사이드 메뉴도 함께 주문하시겠어요?'